# *YOLO* 26s (200 EPOCHS, 1280 IMGSZ)

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

# Load YOLO26n model
model = YOLO("yolo26s.pt")

In [ ]:
# Train the model on your custom dataset from Google Drive
results = model.train(data="/content/drive/MyDrive/DATASET 3 - Pork Rasher/data.yaml", epochs=200, imgsz=1280)

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

# Path to your YOLO training results
results_dir = "/content/runs/detect/train"

# Evaluation plots to display
plot_files = [
    "results.png",
    "BoxF1_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "BoxPR_curve.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png"
]

# Create a figure with 2 columns
fig, axes = plt.subplots(4, 2, figsize=(18, 24))
axes = axes.flatten()

for i, filename in enumerate(plot_files):
    filepath = os.path.join(results_dir, filename)

    if os.path.exists(filepath):
        img = Image.open(filepath)
        axes[i].imshow(img)
        axes[i].set_title(
            filename.replace("_", " ").replace(".png", ""),
            fontsize=14,
            fontweight="bold"
        )
    else:
        axes[i].text(
            0.5, 0.5,
            f"{filename}\nNot found",
            ha="center",
            va="center"
        )

    axes[i].axis("off")

# Hide unused subplot
for j in range(len(plot_files), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
from ultralytics import YOLO

model = YOLO("/content/runs/detect/train/weights/best.pt")

network = model.model
network.eval()

device = next(network.parameters()).device

# Use a dummy image of size 640x640
dummy_input = torch.zeros(
    1, 3, 640, 640,
    device=device
)

layer_info = []
hooks = []

def make_hook(name):
    def hook(module, inputs, output):
        if isinstance(output, torch.Tensor):
            shape = tuple(output.shape)

            # Number of activations per image
            activations = output[0].numel()

            layer_info.append({
                "Layer": name,
                "Type": type(module).__name__,
                "Output Shape": shape,
                "Activations": activations
            })
    return hook

# Register hooks on leaf modules
for name, module in network.named_modules():
    if name and len(list(module.children())) == 0:
        hooks.append(
            module.register_forward_hook(make_hook(name))
        )

# Perform forward pass
with torch.no_grad():
    _ = network(dummy_input)

# Remove hooks
for hook in hooks:
    hook.remove()

# Display results
import pandas as pd

df = pd.DataFrame(layer_info)

display(df)

print("Total recorded activations:",
      f"{df['Activations'].sum():,}")

In [ ]:
import shutil
from google.colab import files

# Zip the folder /content/runs into runs.zip
shutil.make_archive("/content/runs", "zip", "/content/runs")

# Download the zipped file
files.download("/content/runs.zip")

In [ ]:
from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Define the paths
source_path = '/content/runs/detect/train/weights/best.pt'
dest_folder = '/content/drive/MyDrive/YOLO26n Saved Weight'
dest_path = os.path.join(dest_folder, 'best.pt')

# Create the destination directory if it doesn't exist
os.makedirs(dest_folder, exist_ok=True)

# Copy the file to Google Drive
if os.path.exists(source_path):
    shutil.copy(source_path, dest_path)
    print(f"Successfully saved '{source_path}' to '{dest_path}'")
else:
    print(f"Error: Source file '{source_path}' not found. Please verify the path.")